# Joseph's Indicators — Sections 6b & 6d
## Industrial–Logistics Coupling in Otay Mesa | San Diego Border Zone

**Input:** `industrial_businesses_core_geocoded.csv` (from Christian's geocoding script)

**Indicator 1:** Distance from each industrial business point → nearest Port of Entry  
**Indicator 2:** Whether each industrial business lies inside freight-corridor buffers (0.25 / 0.5 / 1 mi)

> Data fetched live from SANDAG FreightViewer.  
> CRS: **EPSG:2230** — CA State Plane Zone VI (US feet) for all distance/buffer math.


## A. Imports & Configuration

In [1]:
import requests
import geopandas as gpd
import pandas as pd
import numpy as np
from shapely.geometry import Point
from io import BytesIO
import warnings
warnings.filterwarnings("ignore")

# ── CRS ─────────────────────────────────────────────────────────────────────
CRS_GEO  = "EPSG:4326"   # geographic — lat/lon
CRS_PROJ = "EPSG:2230"   # CA State Plane Zone VI, US feet — for distance & buffer

# ── Buffer distances in feet ─────────────────────────────────────────────────
BUFFER_DISTANCES_FT = {
    "buffer_quarter_mi": 1320,   # 0.25 mi
    "buffer_half_mi":    2640,   # 0.50 mi
    "buffer_one_mi":     5280,   # 1.00 mi
}

# ── Input file ───────────────────────────────────────────────────────────────
# Place the CSV in the same directory as this notebook, or adjust the path
INPUT_CSV = "industrial_businesses_core_geocoded.csv"

# ── Otay Mesa subarea label values (matches 'city' or a subarea column) ──────
# We'll derive subarea from lat/lon using planning districts — see Section D
OTAY_MESA_DISTRICTS = ["Otay Mesa", "Otay Mesa-Nestor"]

print("Config ready.")


Config ready.


## B. Load Industrial Business Points
Read the geocoded CSV and convert to a GeoDataFrame of points.


In [2]:
df = pd.read_csv(INPUT_CSV, dtype=str)
print(f"Loaded {len(df)} rows")
print("Columns:", df.columns.tolist())
df.head(3)


Loaded 3067 rows
Columns: ['business_acctnum', 'dba_name', 'ownership_type', 'address', 'city', 'state', 'zip', 'naics', 'activity_desc', 'industry_group', 'definition', 'full_address', 'matched_address', 'geocode_score', 'longitude', 'latitude']


,business_acctnum,dba_name,ownership_type,address,city,state,zip,naics,activity_desc,industry_group,definition,full_address,matched_address,geocode_score,longitude,latitude
0,2021003855,098 ENTERPRISES,CORP,3412 LITTLE FLOWER ST,SAN DIEGO,CA,92104-5225,422,"WHOLESALE TRADE, NONDURABLE GOODS",Wholesale Trade,Core,"3412 LITTLE FLOWER ST , SAN DIEGO, CA 92104-5225","2038 Corte del Nogal, Suite 134, Carlsbad, Cal...",100.0,-117.279002413749,33.118762156516
1,1998011168,1 SPIRIT,CORP,3830 VALLEY CENTRE DR 705-903,SAN DIEGO,CA,92130-3320,42282,WINE & DISTILLED ALCOHOLIC BEVERAGE WHSLE,Wholesale Trade,Core,"3830 VALLEY CENTRE DR 705-903 , SAN DIEGO, CA ...","10792 Roselle St, San Diego, California, 92121",100.0,-117.222966541174,32.899788288819
2,2003009618,1-800-GOTJUNK OF SAN DIEGO,LLC,2038 CORTE DEL NOGAL 134,CARLSBAD,CA,92011-1478,488999,ALL OTHER TRANSPORTATION SUPPORT ACTIVITIES,Transportation and Warehousing,Core,"2038 CORTE DEL NOGAL 134 , CARLSBAD, CA 92011-...","3412 Little Flower St, San Diego, California, ...",100.0,-117.119854212329,32.736866931774


In [3]:
# Convert latitude/longitude to numeric and drop any rows missing coords
df["longitude"] = pd.to_numeric(df["longitude"], errors="coerce")
df["latitude"]  = pd.to_numeric(df["latitude"],  errors="coerce")
df = df.dropna(subset=["longitude", "latitude"]).reset_index(drop=True)
print(f"Rows with valid coordinates: {len(df)}")

# Build GeoDataFrame — points in geographic CRS first
businesses = gpd.GeoDataFrame(
    df,
    geometry=gpd.points_from_xy(df["longitude"], df["latitude"]),
    crs=CRS_GEO
)

# Reproject to projected CRS for distance/buffer work
businesses = businesses.to_crs(CRS_PROJ)
print(f"GeoDataFrame CRS: {businesses.crs}")
businesses[["dba_name", "industry_group", "city", "latitude", "longitude"]].head(5)


Rows with valid coordinates: 3067
GeoDataFrame CRS: EPSG:2230


,dba_name,industry_group,city,latitude,longitude
0,098 ENTERPRISES,Wholesale Trade,SAN DIEGO,33.118762,-117.279002
1,1 SPIRIT,Wholesale Trade,SAN DIEGO,32.899788,-117.222967
2,1-800-GOTJUNK OF SAN DIEGO,Transportation and Warehousing,CARLSBAD,32.736867,-117.119854
3,100% SPEEDLAB LLC,Wholesale Trade,SAN DIEGO,32.745302,-117.198323
4,102 SCONE CO.,Manufacturing,LA JOLLA,32.937932,-117.230644


## B2. Clip to San Diego County Boundary (Census TIGER)
Fetches 2023 Census TIGER tract boundaries for California, filters to
San Diego County (FIPS 073), dissolves into a single county polygon,
and clips business points to remove any records outside the county.


In [4]:
# ── Fetch SD County boundary from Census TIGER (FIPS 073 = San Diego) ─────────
TRACT_URL = "https://www2.census.gov/geo/tiger/GENZ2023/shp/cb_2023_06_tract_500k.zip"

print("Fetching Census TIGER 2023 tracts for California...")
tracts_ca = gpd.read_file(TRACT_URL)
print(f"Loaded {len(tracts_ca)} CA tracts")

# Filter to San Diego County (COUNTYFP == '073')
tracts_sd = tracts_ca[tracts_ca["COUNTYFP"] == "073"].copy().to_crs(CRS_PROJ)
print(f"San Diego County tracts: {len(tracts_sd)}")

# Dissolve all tracts into a single SD County boundary polygon
sd_boundary = tracts_sd.geometry.unary_union
sd_county   = gpd.GeoDataFrame(geometry=[sd_boundary], crs=CRS_PROJ)

# ── Clip businesses to SD County ──────────────────────────────────────────────
before = len(businesses)
businesses = businesses[
    businesses.geometry.within(sd_boundary)
].copy().reset_index(drop=True)
after = len(businesses)

print(f"\nBusinesses before clip: {before}")
print(f"Businesses after clip:  {after}  (removed {before - after} outside SD County)")


Fetching Census TIGER 2023 tracts for California...
Loaded 9109 CA tracts
San Diego County tracts: 736

Businesses before clip: 3067
Businesses after clip:  2892  (removed 175 outside SD County)


## C. Fetch SANDAG Reference Layers (Live)
Two layers fetched directly from SANDAG FreightViewer — no local files needed:
- **POE points** — San Ysidro & Otay Mesa ports of entry
- **Major Roads** — freight corridor network


In [5]:
def fetch_geojson(url, label):
    print(f"  Fetching {label}...")
    r = requests.get(url, timeout=60)
    r.raise_for_status()
    gdf = gpd.read_file(BytesIO(r.content))
    if gdf.crs is None:
        gdf = gdf.set_crs(CRS_GEO)
    gdf = gdf.to_crs(CRS_PROJ)
    print(f"    → {len(gdf)} features | reprojected to {CRS_PROJ}")
    return gdf

poe_pts     = fetch_geojson(
    "https://gis.sandag.org/FreightViewer/data/poe_points_update.json",
    "SANDAG POE points"
)
major_roads = fetch_geojson(
    "https://gis.sandag.org/FreightViewer/data/MajorRoads_20171002.json",
    "SANDAG Major Roads"
)

print("\nPOE columns:", poe_pts.columns.tolist())
poe_pts.head()


  Fetching SANDAG POE points...
    → 10 features | reprojected to EPSG:2230
  Fetching SANDAG Major Roads...
    → 45 features | reprojected to EPSG:2230

POE columns: ['OBJECTID_1', 'Status', 'Existing', 'Future', 'Port_ID', 'port_name', 'url', 'wait_time', 'mode', 'geometry']


,OBJECTID_1,Status,Existing,Future,Port_ID,port_name,url,wait_time,mode,geometry
0,1,Existing,1,0,250602,Otay Mesa Commercial,https://www.cbp.gov/contact/ports/otay-mesa,https://bwt.cbp.gov/index.html?com=1&pas=1&ped...,Commercial Vehicle,POINT (6350761.598 1781024.687)
1,2,Existing,1,0,250201,Andrade,https://www.cbp.gov/contact/ports/andrade-class,https://bwt.cbp.gov/index.html?com=1&pas=1&ped...,"Passenger Vehicle, Pedestrian",POINT (7029714.359 1844608.768)
2,3,Existing,1,0,250601,Otay Mesa,https://www.cbp.gov/contact/ports/otay-mesa,https://bwt.cbp.gov/index.html?com=1&pas=1&ped...,"Passenger Vehicle, Pedestrian",POINT (6349476.293 1780798.015)
3,4,Existing,1,0,250302,Calexico West,https://www.cbp.gov/contact/ports/calexico-wes...,https://bwt.cbp.gov/index.html?com=1&pas=1&ped...,"Passenger Vehicle, Pedestrian",POINT (6793345.954 1822588.491)
4,5,Existing,1,0,250301,Calexico East,https://www.cbp.gov/contact/ports/calexico-eas...,https://bwt.cbp.gov/index.html?com=1&pas=1&ped...,"Passenger Vehicle, Commerical Vehicle, Pedestrian",POINT (6826871.094 1826367.536)


## D. Subarea Assignment — Otay Mesa vs. Non-Border Comparison
Network restrictions block the City of SD planning districts layer in this environment.
Subarea is assigned via a hardcoded Otay Mesa bounding box applied **after** the county clip.


In [6]:
# Planning districts fetch was blocked by network restrictions in this environment.
# Subarea assignment uses the hardcoded Otay Mesa bounding box instead.
planning_districts = None


In [7]:
# ── Re-apply subarea assignment on clipped businesses ────────────────────────
# (planning_districts fetch failed, so we use the bounding box fallback)
# This must run AFTER the clip so subarea counts reflect clipped data.

from shapely.geometry import box as shapely_box

if planning_districts is None:
    otay_box = gpd.GeoDataFrame(
        {"subarea": ["Otay Mesa"]},
        geometry=[shapely_box(-117.065, 32.545, -116.930, 32.610)],
        crs=CRS_GEO
    ).to_crs(CRS_PROJ)

    businesses["subarea"] = "Non-Border Comparison"
    businesses.loc[
        businesses.geometry.within(otay_box.geometry.iloc[0]), "subarea"
    ] = "Otay Mesa"
    print("Subarea applied via Otay Mesa bounding box (post-clip).")
else:
    # If planning_districts somehow loaded, do the spatial join
    dist_col = next(
        (c for c in planning_districts.columns
         if any(k in c.upper() for k in ["NAME", "CPNAME", "DISTRICT", "PLAN", "COMM"])),
        planning_districts.columns[0]
    )
    planning_districts["subarea"] = planning_districts[dist_col].apply(
        lambda x: "Otay Mesa"
        if any(d.lower() in str(x).lower() for d in OTAY_MESA_DISTRICTS)
        else "Non-Border Comparison"
    )
    businesses = gpd.sjoin(
        businesses,
        planning_districts[["geometry", "subarea"]],
        how="left", predicate="within"
    ).drop(columns=["index_right"], errors="ignore")
    businesses["subarea"] = businesses["subarea"].fillna("Non-Border Comparison")

print("\nFinal subarea counts (post-clip):")
print(businesses["subarea"].value_counts())
print(f"Total businesses in SD County: {len(businesses)}")


Subarea applied via Otay Mesa bounding box (post-clip).

Final subarea counts (post-clip):
subarea
Non-Border Comparison    2426
Otay Mesa                 466
Name: count, dtype: int64
Total businesses in SD County: 2892


## E. Indicator 1 — Distance to Nearest Port of Entry

For each business point, compute:
- `dist_nearest_poe_ft` / `dist_nearest_poe_mi` — distance to the closer of the two POEs
- `dist_san_ysidro_ft/mi` — distance to San Ysidro POE specifically
- `dist_otay_poe_ft/mi` — distance to Otay Mesa POE specifically
- `nearest_poe_name` — which POE is closest

All distances in US feet (EPSG:2230), converted to miles for readability.


In [8]:
# Identify POE name column
poe_name_col = next(
    (c for c in poe_pts.columns if "name" in c.lower() or "port" in c.lower()),
    None
)
print("POE name column:", poe_name_col)
print("All POE entries:")
display(poe_pts[[poe_name_col, "geometry"]].to_wkt() if poe_name_col else poe_pts)


POE name column: Port_ID
All POE entries:


,Port_ID,geometry
0,250602,POINT (6350761.598184 1781024.6874)
1,250201,POINT (7029714.358757 1844608.767793)
2,250601,POINT (6349476.293055 1780798.014837)
3,250302,POINT (6793345.954313 1822588.491117)
4,250301,POINT (6826871.094253 1826367.535977)
5,250501,POINT (6445369.630467 1789756.856828)
6,250409,POINT (6338334.759564 1779988.643097)
7,250407,POINT (6319429.452558 1778025.749713)
8,250401,POINT (6321460.360898 1778263.364395)
9,0,POINT (6359455.500333 1781687.125333)


In [9]:
# ── Inspect POE layer first ──────────────────────────────────────────────────
print("POE columns:", poe_pts.columns.tolist())
print("\nPOE data types:")
print(poe_pts.dtypes)
print("\nAll POE rows:")
display(poe_pts.drop(columns=["geometry"]))


POE columns: ['OBJECTID_1', 'Status', 'Existing', 'Future', 'Port_ID', 'port_name', 'url', 'wait_time', 'mode', 'geometry']

POE data types:
OBJECTID_1       int32
Status          object
Existing         int32
Future           int32
Port_ID          int32
port_name       object
url             object
wait_time       object
mode            object
geometry      geometry
dtype: object

All POE rows:


,OBJECTID_1,Status,Existing,Future,Port_ID,port_name,url,wait_time,mode
0,1,Existing,1,0,250602,Otay Mesa Commercial,https://www.cbp.gov/contact/ports/otay-mesa,https://bwt.cbp.gov/index.html?com=1&pas=1&ped...,Commercial Vehicle
1,2,Existing,1,0,250201,Andrade,https://www.cbp.gov/contact/ports/andrade-class,https://bwt.cbp.gov/index.html?com=1&pas=1&ped...,"Passenger Vehicle, Pedestrian"
2,3,Existing,1,0,250601,Otay Mesa,https://www.cbp.gov/contact/ports/otay-mesa,https://bwt.cbp.gov/index.html?com=1&pas=1&ped...,"Passenger Vehicle, Pedestrian"
3,4,Existing,1,0,250302,Calexico West,https://www.cbp.gov/contact/ports/calexico-wes...,https://bwt.cbp.gov/index.html?com=1&pas=1&ped...,"Passenger Vehicle, Pedestrian"
4,5,Existing,1,0,250301,Calexico East,https://www.cbp.gov/contact/ports/calexico-eas...,https://bwt.cbp.gov/index.html?com=1&pas=1&ped...,"Passenger Vehicle, Commerical Vehicle, Pedestrian"
5,6,Existing,1,0,250501,Tecate,https://www.cbp.gov/contact/ports/tecate-class,https://bwt.cbp.gov/index.html?com=1&pas=1&ped...,"Passenger Vehicle, Commerical Vehicle, Pedestrian"
6,7,Existing,1,0,250409,San Ysidro Cross Border Express (CBX),https://www.crossborderxpress.com/node/1,https://bwt.cbp.gov/index.html?com=1&pas=1&ped...,Pedestrian
7,8,Existing,1,0,250407,San Ysidro PedWest,https://www.cbp.gov/contact/ports/san-ysidro-c...,https://bwt.cbp.gov/index.html?com=1&pas=1&ped...,Pedestrian
8,9,Existing,1,0,250401,San Ysidro,https://www.cbp.gov/contact/ports/san-ysidro-c...,https://bwt.cbp.gov/index.html?com=1&pas=1&ped...,"Passenger Vehicle, Pedestrian"
9,10,Proposed,0,1,0,Otay Mesa East,N/A,https://www.cbp.gov/contact/ports/otay-mesa,TBD


In [10]:
# ── Distance to each individual POE ─────────────────────────────────────────
# Force name column to string to avoid AttributeError
if poe_name_col:
    poe_pts[poe_name_col] = poe_pts[poe_name_col].astype(str)
    print("POE names (as string):", poe_pts[poe_name_col].unique())

    san_ysidro_geom = poe_pts[
        poe_pts[poe_name_col].str.contains("Ysidro|ysidro|SAN YSIDRO|san ysidro", na=False, case=False)
    ].geometry.unary_union

    otay_mesa_geom = poe_pts[
        poe_pts[poe_name_col].str.contains("Otay|otay|OTAY", na=False, case=False)
    ].geometry.unary_union

    print("San Ysidro geometry found:", san_ysidro_geom is not None)
    print("Otay Mesa geometry found:", otay_mesa_geom is not None)

    if san_ysidro_geom:
        businesses["dist_san_ysidro_ft"] = businesses.geometry.distance(san_ysidro_geom)
        businesses["dist_san_ysidro_mi"] = businesses["dist_san_ysidro_ft"] / 5280
        print("✓ San Ysidro distances computed")
    else:
        # Hardcode San Ysidro POE location as fallback
        print("  San Ysidro not matched by name — using hardcoded coordinates")
        from shapely.geometry import Point
        sy_pt = gpd.GeoDataFrame(
            geometry=[Point(-117.0295, 32.5440)], crs=CRS_GEO
        ).to_crs(CRS_PROJ).geometry.iloc[0]
        businesses["dist_san_ysidro_ft"] = businesses.geometry.distance(sy_pt)
        businesses["dist_san_ysidro_mi"] = businesses["dist_san_ysidro_ft"] / 5280
        print("✓ San Ysidro distances computed (hardcoded coords)")

    if otay_mesa_geom:
        businesses["dist_otay_poe_ft"] = businesses.geometry.distance(otay_mesa_geom)
        businesses["dist_otay_poe_mi"] = businesses["dist_otay_poe_ft"] / 5280
        print("✓ Otay Mesa POE distances computed")
    else:
        # Hardcode Otay Mesa POE location as fallback
        print("  Otay Mesa not matched by name — using hardcoded coordinates")
        from shapely.geometry import Point
        om_pt = gpd.GeoDataFrame(
            geometry=[Point(-116.9447, 32.5726)], crs=CRS_GEO
        ).to_crs(CRS_PROJ).geometry.iloc[0]
        businesses["dist_otay_poe_ft"] = businesses.geometry.distance(om_pt)
        businesses["dist_otay_poe_mi"] = businesses["dist_otay_poe_ft"] / 5280
        print("✓ Otay Mesa POE distances computed (hardcoded coords)")

else:
    # poe_name_col was None — hardcode both POEs directly
    print("No POE name column found — using hardcoded coordinates for both POEs")
    from shapely.geometry import Point

    sy_pt = gpd.GeoDataFrame(
        geometry=[Point(-117.0295, 32.5440)], crs=CRS_GEO
    ).to_crs(CRS_PROJ).geometry.iloc[0]
    om_pt = gpd.GeoDataFrame(
        geometry=[Point(-116.9447, 32.5726)], crs=CRS_GEO
    ).to_crs(CRS_PROJ).geometry.iloc[0]

    businesses["dist_san_ysidro_ft"] = businesses.geometry.distance(sy_pt)
    businesses["dist_san_ysidro_mi"] = businesses["dist_san_ysidro_ft"] / 5280
    businesses["dist_otay_poe_ft"]   = businesses.geometry.distance(om_pt)
    businesses["dist_otay_poe_mi"]   = businesses["dist_otay_poe_ft"] / 5280
    print("✓ Both POE distances computed via hardcoded coordinates")

# ── Distance to nearest POE (whichever is closer) ────────────────────────────
poe_union = poe_pts.geometry.unary_union
businesses["dist_nearest_poe_ft"] = businesses.geometry.distance(poe_union)
businesses["dist_nearest_poe_mi"] = businesses["dist_nearest_poe_ft"] / 5280

# ── Which POE is nearest ──────────────────────────────────────────────────────
if "dist_san_ysidro_ft" in businesses.columns and "dist_otay_poe_ft" in businesses.columns:
    businesses["nearest_poe_name"] = np.where(
        businesses["dist_san_ysidro_ft"] <= businesses["dist_otay_poe_ft"],
        "San Ysidro", "Otay Mesa"
    )

print("\nDistance columns added:")
print([c for c in businesses.columns if "dist_" in c or "nearest_poe" in c])


POE names (as string): ['250602' '250201' '250601' '250302' '250301' '250501' '250409' '250407'
 '250401' '0']
San Ysidro geometry found: True
Otay Mesa geometry found: True
  San Ysidro not matched by name — using hardcoded coordinates
✓ San Ysidro distances computed (hardcoded coords)
  Otay Mesa not matched by name — using hardcoded coordinates
✓ Otay Mesa POE distances computed (hardcoded coords)

Distance columns added:
['dist_san_ysidro_ft', 'dist_san_ysidro_mi', 'dist_otay_poe_ft', 'dist_otay_poe_mi', 'dist_nearest_poe_ft', 'dist_nearest_poe_mi', 'nearest_poe_name']


In [11]:
# ── Summary statistics by subarea ────────────────────────────────────────────
print("\n─── Distance to Nearest POE (miles) by Subarea ───")
summary_poe = (
    businesses
    .groupby("subarea")["dist_nearest_poe_mi"]
    .agg(n="count", mean="mean", median="median", std="std", min="min", max="max")
    .round(3)
)
display(summary_poe)

if "nearest_poe_name" in businesses.columns:
    print("\n─── Which POE is Nearest, by Subarea ───")
    display(
        businesses.groupby(["subarea", "nearest_poe_name"]).size()
        .reset_index(name="count")
    )



─── Distance to Nearest POE (miles) by Subarea ───


,n,mean,median,std,min,max
subarea,,,,,,
Non-Border Comparison,2426,20.115,20.066,8.338,0.124,59.582
Otay Mesa,466,1.214,1.060,0.859,0.118,4.393



─── Which POE is Nearest, by Subarea ───


,subarea,nearest_poe_name,count
0,Non-Border Comparison,Otay Mesa,742
1,Non-Border Comparison,San Ysidro,1684
2,Otay Mesa,Otay Mesa,313
3,Otay Mesa,San Ysidro,153


## F. Indicator 2 — Freight-Corridor Buffer Overlap

Major freight roads are buffered at **0.25 mi, 0.5 mi, and 1 mi**.  
For each business point, a binary flag indicates whether it falls inside each buffer zone.

> Since these are points (not polygons), the indicator is simply **inside / outside** —
> no area-share calculation needed.


In [12]:
# Dissolve all road segments into one geometry
roads_dissolved = major_roads.dissolve().geometry.iloc[0]
print("Major Roads dissolved into single geometry.")
print("Road columns:", major_roads.columns.tolist())


Major Roads dissolved into single geometry.
Road columns: ['OBJECTID_1', 'OBJECTID', 'IFC', 'LABEL', 'EXISTING', 'Shape_Leng', 'Shape_Le_1', 'NM', 'Source', 'Mileage', 'geometry']


In [13]:
# Compute buffer flags for each distance
for buf_col, buf_ft in BUFFER_DISTANCES_FT.items():
    buf_mi = buf_ft / 5280
    corridor_buffer = roads_dissolved.buffer(buf_ft)
    businesses[f"{buf_col}"] = businesses.geometry.within(corridor_buffer)
    n_in  = businesses[f"{buf_col}"].sum()
    total = len(businesses)
    print(f"  {buf_mi:.2f}-mile buffer: {n_in}/{total} businesses inside ({100*n_in/total:.1f}%)")


  0.25-mile buffer: 830/2892 businesses inside (28.7%)
  0.50-mile buffer: 1541/2892 businesses inside (53.3%)
  1.00-mile buffer: 2102/2892 businesses inside (72.7%)


In [14]:
# ── Summary by subarea ────────────────────────────────────────────────────────
for buf_col, buf_ft in BUFFER_DISTANCES_FT.items():
    buf_mi = buf_ft / 5280
    print(f"\n─── {buf_mi:.2f}-mile freight corridor buffer ───")
    summary = (
        businesses
        .groupby("subarea")
        .agg(
            n_businesses = ("dba_name",  "count"),
            n_inside     = (buf_col,     "sum"),
            pct_inside   = (buf_col,     lambda x: round(100 * x.mean(), 1)),
        )
    )
    display(summary)



─── 0.25-mile freight corridor buffer ───


,n_businesses,n_inside,pct_inside
subarea,,,
Non-Border Comparison,2426,616,25.4
Otay Mesa,466,214,45.9



─── 0.50-mile freight corridor buffer ───


,n_businesses,n_inside,pct_inside
subarea,,,
Non-Border Comparison,2426,1147,47.3
Otay Mesa,466,394,84.5



─── 1.00-mile freight corridor buffer ───


,n_businesses,n_inside,pct_inside
subarea,,,
Non-Border Comparison,2426,1653,68.1
Otay Mesa,466,449,96.4


## G. Export Results

Two outputs:
- **`joseph_indicators_output.csv`** — flat table (lat/lon preserved) for ArcGIS import
- **`joseph_indicators_output.geojson`** — point layer with all indicator columns for StoryMap


In [15]:
# Columns to keep in output
base_cols = [
    "business_acctnum", "dba_name", "industry_group", "naics",
    "address", "city", "state", "zip",
    "latitude", "longitude", "subarea",
    "dist_nearest_poe_ft", "dist_nearest_poe_mi",
]
optional_cols = [
    "dist_san_ysidro_ft", "dist_san_ysidro_mi",
    "dist_otay_poe_ft",   "dist_otay_poe_mi",
    "nearest_poe_name",
]
buffer_cols  = list(BUFFER_DISTANCES_FT.keys())

keep = (
    base_cols
    + [c for c in optional_cols if c in businesses.columns]
    + buffer_cols
)

output_gdf = businesses[[c for c in keep if c in businesses.columns]
                         + ["geometry"]].copy()

# ── CSV ───────────────────────────────────────────────────────────────────────
csv_out = output_gdf.drop(columns=["geometry"])
csv_out.to_csv("joseph_indicators_output.csv", index=False)
print("✓ Exported: joseph_indicators_output.csv")
print(f"  {len(csv_out)} rows, {len(csv_out.columns)} columns")

# ── GeoJSON (reproject back to WGS84 for web/StoryMap) ───────────────────────
output_gdf.to_crs("EPSG:4326").to_file(
    "joseph_indicators_output.geojson", driver="GeoJSON"
)
print("✓ Exported: joseph_indicators_output.geojson")


✓ Exported: joseph_indicators_output.csv
  2892 rows, 21 columns
✓ Exported: joseph_indicators_output.geojson


In [16]:
# ── Final validation table ────────────────────────────────────────────────────
print("\n─── FINAL VALIDATION SUMMARY ───")
val_cols = ["subarea", "dist_nearest_poe_mi"] + buffer_cols
display(
    businesses[[c for c in val_cols if c in businesses.columns]]
    .groupby("subarea")
    .agg(["mean", "count"])
    .round(3)
)
print("\nAll done. Load joseph_indicators_output.geojson into ArcGIS for the StoryMap.")



─── FINAL VALIDATION SUMMARY ───


dist_nearest_poe_mi       buffer_quarter_mi        \
                                     mean count              mean count   
subarea                                                                   
Non-Border Comparison              20.115  2426             0.254  2426   
Otay Mesa                           1.214   466             0.459   466   

                      buffer_half_mi       buffer_one_mi        
                                mean count          mean count  
subarea                                                         
Non-Border Comparison          0.473  2426         0.681  2426  
Otay Mesa                      0.845   466         0.964   466


All done. Load joseph_indicators_output.geojson into ArcGIS for the StoryMap.
